# Alfvén Radius From Shell Sampling

`r_A` is defined here as the first outward radial distance where `M_A` changes from `<1` to `>1`.
This approximates the 3D `M_A=1` isosurface by directional shell/ray sampling.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from batwind import SmartDs
from batwind.analysis.shells import sample_spherical_shells
from batwind.data.samples import data_file
from batwind.physics.alfven_radius import alfven_radius_map
from batwind.physics.alfven_radius import projected_solid_angle_weights
from batwind.physics.alfven_radius import summarize_alfven_radius


In [ ]:
data_path = data_file("3d__var_4_n00000000.plt")
sds = SmartDs.from_file(data_path)
star_radius_m = float(sds["RBODY [m]"])
print(sds)
print(f"RBODY [m]: {star_radius_m:.6e}")


In [ ]:
r_all = sds["R [R]"]
r_max = float(np.nanmax(r_all))

n_radii = 96
n_polar = 64
n_azimuth = 128

radii = np.geomspace(1.0, r_max, n_radii)
print(f"R grid [R]: {radii[0]:.3g} -> {radii[-1]:.3g}")
print(f"n_radii={n_radii}, n_polar={n_polar}, n_azimuth={n_azimuth}")

shell_ds = sample_spherical_shells(
    sds,
    radii,
    fields=("M_A [none]",),
    n_polar=n_polar,
    n_azimuth=n_azimuth,
    method="nearest",
    length_unit_to_m=star_radius_m,
)

ra_map = alfven_radius_map(shell_ds)
weights = projected_solid_angle_weights(shell_ds)
polar_map = shell_ds["polar [rad]"][0]

min_ra, max_ra, avg_ra, avg_ra_cyl, coverage = summarize_alfven_radius(
    ra_map,
    polar_map,
    weights=weights,
)

print(f"Min r_A [R]: {min_ra:.3f}")
print(f"Max r_A [R]: {max_ra:.3f}")
print(f"Avg r_A [R]: {avg_ra:.3f}")
print(f"Avg cylindrical r_A [R]: {avg_ra_cyl:.3f}")
print(f"Coverage [none]: {coverage:.3f}")


In [ ]:
longitude = np.degrees(shell_ds["azimuth [rad]"][0])
latitude = 90.0 - np.degrees(shell_ds["polar [rad]"][0])

fig, ax = plt.subplots(figsize=(9.5, 5.2))
img = ax.pcolormesh(longitude, latitude, ra_map, shading="nearest", cmap="viridis")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Latitude [deg]")
ax.set_title("Alfvén Radius r_A [R]")
fig.colorbar(img, ax=ax, label="r_A [R]")
plt.show()
